# Project 7: Development Permit Approval Predictor -- Exploratory Data Analysis

This notebook explores the Calgary Development Permits dataset (~188,653 records, 40 columns) to
understand the data distribution, temporal trends, permit-status breakdown, description text
characteristics, and community-level patterns before building the approval prediction model.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Make src importable
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_and_preprocess, fetch_data

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

print("Imports loaded successfully.")

## 1. Data Loading

In [ ]:
# Load raw data
raw = fetch_data(use_cache=True)
print(f"Raw dataset: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(3)

In [ ]:
# Preprocess
df = load_and_preprocess(use_cache=True)
print(f"Preprocessed dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

In [ ]:
# Basic info
df.info()

In [ ]:
# Missing values summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing": missing, "Pct": missing_pct})
missing_df[missing_df["Missing"] > 0].sort_values("Pct", ascending=False)

## 2. Status Distribution (Target Variable)

In [ ]:
# Current status value counts
status_counts = df["statuscurrent"].value_counts().reset_index()
status_counts.columns = ["Status", "Count"]
print(f"Unique statuses: {len(status_counts)}")
status_counts

In [ ]:
# Pie chart of top statuses
fig = px.pie(
    status_counts.head(10),
    names="Status",
    values="Count",
    title="Development Permits by Current Status (Top 10)",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.update_layout(width=700, height=450)
fig.show()

In [ ]:
# Binary target distribution
target_counts = df["approved"].value_counts().reset_index()
target_counts.columns = ["Approved", "Count"]
target_counts["Approved"] = target_counts["Approved"].map({1: "Approved", 0: "Not Approved"})

fig = px.bar(
    target_counts,
    x="Approved",
    y="Count",
    color="Approved",
    color_discrete_map={"Approved": "#2ecc71", "Not Approved": "#e74c3c"},
    title="Binary Target Distribution: Approved vs Not Approved",
    text_auto=True,
)
fig.update_layout(width=600, height=400, showlegend=False)
fig.show()

print(f"Approval rate: {df['approved'].mean():.2%}")

## 3. Temporal Trends

In [ ]:
# Permits per year
yearly = df.groupby("applied_year").agg(
    total=("approved", "count"),
    approved=("approved", "sum"),
).reset_index()
yearly["approval_rate"] = yearly["approved"] / yearly["total"]
yearly = yearly[yearly["applied_year"] > 1979]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=yearly["applied_year"], y=yearly["total"], name="Total Permits", marker_color="steelblue"),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=yearly["applied_year"], y=yearly["approval_rate"], name="Approval Rate",
               mode="lines+markers", marker_color="green"),
    secondary_y=True,
)
fig.update_layout(title="Permits per Year & Approval Rate", width=900, height=450)
fig.update_yaxes(title_text="Number of Permits", secondary_y=False)
fig.update_yaxes(title_text="Approval Rate", tickformat=".0%", secondary_y=True)
fig.show()

In [ ]:
# Permits by month
monthly = df.groupby("applied_month").size().reset_index(name="Count")
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly["Month"] = monthly["applied_month"].apply(lambda m: month_labels[int(m)-1] if 1 <= m <= 12 else "?")

fig = px.bar(
    monthly, x="Month", y="Count",
    title="Permit Applications by Month",
    color="Count", color_continuous_scale="Viridis",
)
fig.update_layout(width=700, height=400)
fig.show()

In [ ]:
# Permits by day of week
dow = df.groupby("applied_day_of_week").size().reset_index(name="Count")
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
dow["Day"] = dow["applied_day_of_week"].apply(lambda d: dow_labels[int(d)] if 0 <= d <= 6 else "?")

fig = px.bar(
    dow, x="Day", y="Count",
    title="Permit Applications by Day of Week",
    color="Count", color_continuous_scale="Sunset",
)
fig.update_layout(width=600, height=400)
fig.show()

## 4. Description Text Analysis

In [ ]:
# Description length statistics
df["desc_length"] = df["description_clean"].fillna("").str.len()
df["desc_word_count"] = df["description_clean"].fillna("").str.split().str.len()

print("Description length (characters):")
print(df["desc_length"].describe().round(1))
print("\nDescription word count:")
print(df["desc_word_count"].describe().round(1))

In [ ]:
# Distribution of description lengths by approval status
fig = px.histogram(
    df[df["desc_length"] > 0],
    x="desc_word_count",
    color=df[df["desc_length"] > 0]["approved"].map({1: "Approved", 0: "Not Approved"}),
    nbins=50,
    barmode="overlay",
    opacity=0.6,
    title="Distribution of Description Word Count by Approval Status",
    labels={"desc_word_count": "Word Count", "color": "Status"},
    color_discrete_map={"Approved": "#2ecc71", "Not Approved": "#e74c3c"},
)
fig.update_layout(width=800, height=400)
fig.show()

In [ ]:
# Most common words overall
from sklearn.feature_extraction.text import CountVectorizer

corpus = df["description_clean"].fillna("").astype(str)
cv = CountVectorizer(stop_words="english", max_features=30)
word_matrix = cv.fit_transform(corpus)
word_counts = pd.DataFrame({
    "Word": cv.get_feature_names_out(),
    "Frequency": word_matrix.sum(axis=0).A1,
}).sort_values("Frequency", ascending=False)

fig = px.bar(
    word_counts, x="Frequency", y="Word", orientation="h",
    title="Top 30 Words in Permit Descriptions",
    color="Frequency", color_continuous_scale="Tealgrn",
)
fig.update_layout(yaxis=dict(autorange="reversed"), width=700, height=600)
fig.show()

In [ ]:
# Top words for approved vs not-approved
for label, subset in [("Approved", df[df["approved"]==1]), ("Not Approved", df[df["approved"]==0])]:
    cv_sub = CountVectorizer(stop_words="english", max_features=15)
    mat = cv_sub.fit_transform(subset["description_clean"].fillna("").astype(str))
    wc = pd.DataFrame({
        "Word": cv_sub.get_feature_names_out(),
        "Frequency": mat.sum(axis=0).A1,
    }).sort_values("Frequency", ascending=False)
    
    fig = px.bar(
        wc, x="Frequency", y="Word", orientation="h",
        title=f"Top 15 Words -- {label} Permits",
        color="Frequency", color_continuous_scale="Greens" if label == "Approved" else "Reds",
    )
    fig.update_layout(yaxis=dict(autorange="reversed"), width=600, height=400)
    fig.show()

## 5. Community Patterns

In [ ]:
# Top 25 communities by permit volume
top_comm = df["communityname"].value_counts().head(25).reset_index()
top_comm.columns = ["Community", "Permits"]

fig = px.bar(
    top_comm, x="Permits", y="Community", orientation="h",
    title="Top 25 Communities by Permit Volume",
    color="Permits", color_continuous_scale="Tealgrn",
)
fig.update_layout(yaxis=dict(autorange="reversed"), width=800, height=600)
fig.show()

In [ ]:
# Approval rate by top communities
top_25_names = top_comm["Community"].tolist()
comm_approval = (
    df[df["communityname"].isin(top_25_names)]
    .groupby("communityname")["approved"]
    .mean()
    .reset_index()
)
comm_approval.columns = ["Community", "Approval Rate"]
comm_approval.sort_values("Approval Rate", ascending=True, inplace=True)

fig = px.bar(
    comm_approval, x="Approval Rate", y="Community", orientation="h",
    title="Approval Rate by Top 25 Communities",
    color="Approval Rate", color_continuous_scale="RdYlGn",
)
fig.update_layout(yaxis=dict(autorange="reversed"), width=800, height=600)
fig.show()

In [ ]:
# Permits by quadrant
quad = df["quadrant"].value_counts().reset_index()
quad.columns = ["Quadrant", "Count"]

fig = px.pie(
    quad, names="Quadrant", values="Count",
    title="Permits by City Quadrant",
    hole=0.35,
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_layout(width=600, height=400)
fig.show()

In [ ]:
# Permit category distribution
cat = df["category"].value_counts().head(15).reset_index()
cat.columns = ["Category", "Count"]

fig = px.bar(
    cat, x="Count", y="Category", orientation="h",
    title="Top 15 Permit Categories",
    color="Count", color_continuous_scale="Sunset",
)
fig.update_layout(yaxis=dict(autorange="reversed"), width=700, height=450)
fig.show()

In [ ]:
# Approval rate by permit category
cat_approval = (
    df.groupby("category")["approved"]
    .agg(["mean", "count"])
    .reset_index()
)
cat_approval.columns = ["Category", "Approval Rate", "Count"]
cat_approval = cat_approval[cat_approval["Count"] >= 100].sort_values("Approval Rate", ascending=True)

fig = px.bar(
    cat_approval, x="Approval Rate", y="Category", orientation="h",
    title="Approval Rate by Permit Category (min 100 permits)",
    color="Approval Rate", color_continuous_scale="RdYlGn",
    hover_data=["Count"],
)
fig.update_layout(yaxis=dict(autorange="reversed"), width=800, height=500)
fig.show()

In [ ]:
# Permitted vs discretionary
if "permitteddiscretionary" in df.columns:
    pd_counts = df["permitteddiscretionary"].value_counts().reset_index()
    pd_counts.columns = ["Type", "Count"]
    
    fig = px.pie(
        pd_counts, names="Type", values="Count",
        title="Permitted vs Discretionary Permits",
        hole=0.35,
        color_discrete_sequence=px.colors.qualitative.Bold,
    )
    fig.update_layout(width=600, height=400)
    fig.show()
    
    # Approval rate
    pd_approval = df.groupby("permitteddiscretionary")["approved"].mean().reset_index()
    pd_approval.columns = ["Type", "Approval Rate"]
    print("Approval rate by type:")
    print(pd_approval.to_string(index=False))

## 6. Summary of Findings

Key observations from the exploratory analysis:

1. **Target balance** -- The majority of permits are approved, creating a class-imbalance scenario that the model must account for.
2. **Temporal trends** -- Permit volume fluctuates with economic cycles (e.g., oil-price downturns). Approval rates have remained relatively stable over time.
3. **Description text** -- Approved permits tend to use common construction terminology ("house", "garage", "suite"). Descriptions vary in length; very short descriptions may signal incomplete applications.
4. **Community effects** -- Some communities have noticeably higher or lower approval rates, likely reflecting land-use district differences.
5. **Category** -- Certain permit categories (e.g., commercial renovations) have lower approval rates than residential categories.
6. **Permitted vs discretionary** -- Discretionary permits have a measurably lower approval rate than permitted-use applications.

These insights inform feature selection for the classification model.